# 분류

##### - pos_label
- 어떤 클래스를 Positive(양성) 로 볼지 지정하는 옵션
- `f1_score`, `precision_score`, `recall_score` 에서 사용
- `accuracy_score`, `roc_auc_score` 에선 사용 X

```python
f1_score(y_true, y_pred, pos_label="yes") → "yes" 를 Positive class 로 간주하여 계산

##### - roc_auc_score
- ROC 곡선 아래 면적(AUC)을 계산하는 평가지표
- ROC 곡선은 **분류 확률값의 변화**를 기반으로 생성됨
- 따라서 `predict()`가 아닌 `predict_proba()` 사용 권장
- rf.classes_에서 두번째(큰 값)을 positive로 삼음
- 다중 분류일 경우 `roc_auc_score(y_true, predict_proba, multi_class="ovr")`

##### - 분류(Classification)
- train / valid 데이터의 클래스 비율 유지를 위해 보통 `stratify = y` 사용

##### - predict_proba
- 각 클래스에 대한 확률을 보고 싶을 때 사용
- [:, 1] : 2번째 클래스에 해당하는 확률
- 다중 클래스에 대한 확률을 보고 싶을 때는 [:, 1] 이런거 하면 안됨

# RandomizedSearchCV에서 어떤 scoring이 있는지 확인

```
from sklearn.metrics import get_scorer_names
print(sorted(get_scorer_names()))
```

## 서비스 이탈예측 데이터
데이터 설명 : 고객의 신상정보 데이터를 통한 회사 서비스 이탈 예측 (종속변수 : Exited)  
x_train : https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/X_train.csv  
y_train : https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/y_train.csv  
x_test : https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/y_test.csv  
데이터 출처 : https://www.kaggle.com/shubh0799/churn-modelling 에서 변형

In [ ]:
import pandas as pd
#데이터 로드
x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/X_test.csv")

display(x_train.head())
display(y_train.head())
display(x_test.head())

In [ ]:
drop_col = ["CustomerId", "Surname"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["Exited"]

from sklearn.preprocessing import StandardScaler
column = x_train_drop.select_dtypes(exclude = object).columns

scaler = StandardScaler()
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column]) 

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)

x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, )

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMRegressor, LGBMClassifier

TASK = "classifier"

if TASK == "regressor":
    models = {
        "RandomForest" : RandomForestRegressor(random_state = 23),
        "GradientBoosting" : GradientBoostingRegressor(random_state = 23),
        "lightGBM" : LGBMRegressor(random_state = 23)
    }

elif TASK == "classifier":
    models = {
        "RandomForest" : RandomForestClassifier(random_state = 23),
        "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
        "lightGBM" : LGBMClassifier(random_state = 23)
    }

In [ ]:
from sklearn.metrics import root_mean_squared_error, accuracy_score, f1_score, r2_score

for name, model in models.items():
    if TASK == "regressor":
        model.fit(X_train, Y_train)
        pred = model.predict(X_valid)
        rmse = root_mean_squared_error(Y_valid, pred)
        r2 = r2_score(Y_valid, pred)
        print(f"{name} | RMSE : {rmse:.4f}, r2 : {r2:.4f}")

    elif TASK == "classifier":
        model.fit(X_train, Y_train)
        pred = model.predict(X_valid)
        accuracy = accuracy_score(Y_valid, pred)
        f1 = f1_score(Y_valid, pred)
        print(f"{name} | accuracy_score : {accuracy:.4f}, f1-score : {f1:.4f}")

In [ ]:
best_estimator = GradientBoostingClassifier(random_state = 23)
scoring = "accuracy"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("최적의 파라미터 : ", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)
print("train accuracy :", accuracy_score(Y_train, pred_train))
print("valid accuracy :", accuracy_score(Y_valid, pred_valid))
print("train F1   :", f1_score(Y_train, pred_train, average='weighted'))
print("valid F1   :", f1_score(Y_valid, pred_valid, average='weighted'))

In [ ]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred" : pred
})

df

## 이직여부 판단 데이터
데이터 설명 : 이직여부 판단 데이터 (target: 1: 이직 , 0 : 이직 x)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/X_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/y_test.csv  
데이터 출처 :https://www.kaggle.com/datasets/arashnic/hr-analytics-job-change-of-data-scientists (참고, 데이터 수정)

In [ ]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/X_test.csv")

display(x_train.head())
display(y_train.head())

In [ ]:
drop_col = ["enrollee_id"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["target"]


from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude = object).columns

scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23)
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred)
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
best_estimator = LGBMClassifier(random_state = 23)
scoring = "accuracy"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train)
f1_valid = f1_score(Y_valid, pred_valid)

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

In [ ]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

## 정시 배송 여부 판단 (2회기출)
데이터 설명 : e-commerce 배송의 정시 도착여부 (1: 정시배송 0 : 정시미배송)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/X_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/y_test.csv  
데이터 출처 :https://www.kaggle.com/datasets/prachi13/customer-analytics (참고, 데이터 수정)

In [ ]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/X_test.csv")

display(x_train.head())
display(y_train.head())

In [ ]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["Reached.on.Time_Y.N"]


from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude = object).columns

scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23)
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred)
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
best_estimator = GradientBoostingClassifier(random_state = 23)
scoring = "accuracy"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train)
f1_valid = f1_score(Y_valid, pred_valid)

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

In [ ]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

## 성인 건강검진 데이터
데이터 설명 : 2018년도 성인의 건강검 진데이터 (종속변수 : 흡연상태 1- 흡연, 0-비흡연 )  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/x_test.csv x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/y_test.csv  
데이터 출처 :https://www.data.go.kr/data/15007122/fileData.do (참고, 데이터 수정)

In [ ]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/x_test.csv")


display(x_train.head())
display(y_train.head())

In [ ]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["흡연상태"]


from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude = object).columns

scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23)
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred)
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
best_estimator = GradientBoostingClassifier(random_state = 23)
scoring = "accuracy"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train)
f1_valid = f1_score(Y_valid, pred_valid)

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

In [ ]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

## 자동차 보험가입 예측데이터
데이터 설명 : 자동차 보험 가입 예측 (종속변수 Response: 1 : 가입 , 0 :미가입)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/y_test.csv  
데이터 출처 :https://www.kaggle.com/anmolkumar/health-insurance-cross-sell-prediction(참고, 데이터 수정)근처 자동차 대리점

In [ ]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/x_test.csv")

display(x_train.head())
display(y_train.head())

In [ ]:
drop_col = ["ID", "id"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["Response"]


from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude = object).columns

scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23, class_weight = "balanced"),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23, class_weight = "balanced")
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred)
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
best_estimator = LGBMClassifier(random_state = 23)
scoring = "f1"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train)
f1_valid = f1_score(Y_valid, pred_valid)

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

In [ ]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

## ❌ 비행탑승 경험 만족도 데이터
데이터 설명 : 비행탑승 경험 만족도 (satisfaction 컬럼 : ‘neutral or dissatisfied’ or satisfied ) (83123, 24) shape  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/y_test.csv  
데이터 출처 :https://www.kaggle.com/teejmahal20/airline-passenger-satisfaction?select=train.csv (참고, 데이터 수정)

### test 데이터에 대해서 neutral or dissatisfied라고 예측할 확률을 구하고 그 확률 값을 제출하라

In [ ]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/x_test.csv")

display(x_train.head())
display(y_train.head())

In [ ]:
x_train = x_train.fillna(0)
x_test = x_test.fillna(0)

In [ ]:
drop_col = ["ID", "id"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = (y_train["satisfaction"] == "neutral or dissatisfied").astype(int) # roc_auc_score를 이용할 거기 때문에 우리가 원하는 neutral or dissatisfied를 1로 바꿈

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

column = x_train_drop.select_dtypes(exclude = object).columns
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23)
}


from sklearn.metrics import roc_auc_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict_proba(X_valid)[:, 1] # model.classes_ 하면 1번째가 neutral or dissatisfied임
    roc_auc = roc_auc_score(Y_valid, pred)
    print(f"{name} | ROC-AUC : {roc_auc:.4f}")

In [ ]:
best_estimator = LGBMClassifier(random_state = 23)

from sklearn.metrics import get_scorer_names
print(get_scorer_names())

scoring = "roc_auc"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [3, 4, 5],
    "learning_rate" : [0.01, 0.05, 0.1],
    "subsample" : [0.8, 0.9, 1]
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("Best Param :", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

roc_auc_train = roc_auc_score(Y_train, pred_train)
roc_auc_valid = roc_auc_score(Y_valid, pred_valid)


print("roc_auc(train) :", roc_auc_train, "roc_auc(valid) :", roc_auc_valid)

In [ ]:
pred = best_model.predict_proba(x_test_dummies)[:, 1]
df = pd.DataFrame({
    "pred" : pred
})

df

## ❌ 수질 음용성 여부 데이터
데이터 설명 : 수질 음용성 여부 (Potablillity 컬럼 : 0 ,1 )  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/y_test.csv  
데이터 출처 :https://www.kaggle.com/adityakadiwal/water-potability

In [ ]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/x_test.csv")

display(x_train.head())
display(y_train.head())

In [ ]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["Potability"]

x_train_drop = x_train_drop.fillna(x_train_drop.median())
x_test_drop  = x_test_drop.fillna(x_train_drop.median()) 

from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude=object).columns 
scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns=x_train_dummies.columns, fill_value=0)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23)
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred)
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
best_estimator = RandomForestClassifier(random_state = 23)
scoring = "f1"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train)
f1_valid = f1_score(Y_valid, pred_valid)

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

In [ ]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

## 약물 분류 데이터
데이터 설명 : 투약하는 약을 분류 (종속변수 :Drug)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/y_test.csv  
데이터 출처 :https://www.kaggle.com/prathamtripathi/drug-classification(참고, 데이터 수정)

In [ ]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/x_test.csv")

display(x_train.head())
display(y_train.head())

In [ ]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["Drug"]

from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude=object).columns 
scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns=x_train_dummies.columns, fill_value=0)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23)
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred, average="macro")
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
best_estimator = LGBMClassifier(random_state = 23)
scoring = "accuracy"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train, average="macro")
f1_valid = f1_score(Y_valid, pred_valid, average="macro")

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

In [ ]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

## ❌ 사기회사 분류 데이터
데이터 설명 : 사기회사 분류 (종속변수 : Risk 1: 사기 , 0 : 정상)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/y_test.csv  
데이터 출처 :https://www.kaggle.com/sid321axn/audit-data(참고, 데이터 수정)

In [ ]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/x_test.csv")

display(x_train.head())
display(y_train.head())


In [ ]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["Risk"]

from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude = object).columns

scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

x_train_dummies = x_train_dummies.fillna(x_train_dummies.mean())
x_test_dummies  = x_test_dummies.fillna(x_train_dummies.mean())

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23, class_weight = "balanced"),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23, class_weight = "balanced")
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred)
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
best_estimator = GradientBoostingClassifier(random_state = 23)
scoring = "accuracy"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train)
f1_valid = f1_score(Y_valid, pred_valid)

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

In [ ]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

## 센서데이터 동작유형 분류 데이터
데이터 설명 : 센서데이터로 동작 유형 분류 (종속변수 pose : 0 ,1 구분)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/y_test.csv  
데이터 출처 :https://www.kaggle.com/kyr7plus/emg-4(참고, 데이터 수정)

In [1]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,motion_0,motion_1,motion_2,motion_3,motion_4,motion_5,motion_6,motion_7,motion_8,...,motion_54,motion_55,motion_56,motion_57,motion_58,motion_59,motion_60,motion_61,motion_62,motion_63
0,0,1.0,-2.0,-1.0,4.0,-5.0,-4.0,1.0,0.0,-15.0,...,0.0,-1.0,-13.0,-3.0,1.0,-1.0,-32.0,-22.0,-2.0,-3.0
1,2,20.0,0.0,0.0,1.0,5.0,6.0,-52.0,18.0,15.0,...,-70.0,-55.0,-38.0,-14.0,-12.0,-8.0,-34.0,-63.0,-87.0,-77.0
2,4,1.0,-1.0,1.0,4.0,-5.0,-8.0,1.0,-3.0,-14.0,...,1.0,12.0,-25.0,0.0,0.0,3.0,2.0,-27.0,1.0,0.0
3,5,13.0,2.0,1.0,-3.0,1.0,3.0,28.0,3.0,12.0,...,0.0,-21.0,-17.0,-2.0,0.0,-4.0,-17.0,-21.0,-21.0,25.0
4,6,-2.0,-7.0,-4.0,-8.0,16.0,44.0,1.0,3.0,-16.0,...,-1.0,2.0,-1.0,1.0,4.0,4.0,-17.0,-38.0,-3.0,3.0


,ID,pose
0,0,1
1,2,0
2,4,1
3,5,0
4,6,1


In [2]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["pose"]

from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude = object).columns

scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23, class_weight = "balanced"),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23, class_weight = "balanced")
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred)
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

RandomForest | Accuracy : 0.9943, F1-score : 0.9943
GradientBoosting | Accuracy : 0.9978, F1-score : 0.9979
[LightGBM] [Info] Number of positive: 1619, number of negative: 1636
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000384 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6784
[LightGBM] [Info] Number of data points in the train set: 3255, number of used features: 64
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
lightGBM | Accuracy : 0.9978, F1-score : 0.9979


In [4]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [5]:
best_estimator = GradientBoostingClassifier(random_state = 23)
scoring = "accuracy"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

best param : {'subsample': 1.0, 'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.05}


In [6]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train)
f1_valid = f1_score(Y_valid, pred_valid)

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

accuracy score(train) : 1.0 accuracy score(valid) : 0.9978494623655914
f1 score(train) : 1.0 f1 score(valid) : 0.997867803837953


In [7]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

,pred
0,1
1,1
2,1
3,1
4,0
...,...
1158,0
1159,0
1160,1
1161,1


## 당뇨여부판단 데이터
데이터 설명 : 당뇨여부 판단하기 (종속변수 Outcome : 1 당뇨 , 0 :정상)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/y_test.csv  
데이터 출처 :https://www.kaggle.com/pritsheta/diabetes-dataset(참고, 데이터 수정)

In [8]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0,8,126,88,36,108,38.5,0.349,49
1,1,0,74,52,10,36,27.8,0.269,22
2,2,1,140,74,26,180,24.1,0.828,23
3,3,6,162,62,0,0,24.3,0.178,50
4,4,2,94,68,18,76,26.0,0.561,21


,ID,Outcome
0,0,0
1,1,0
2,2,0
3,3,1
4,4,0


In [14]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop  = x_test.drop(columns = drop_col)
y = y_train["Outcome"]

from sklearn.preprocessing import StandardScaler
num_col = x_train_drop.select_dtypes(exclude = object).columns

scaler = StandardScaler()
x_train_drop[num_col] = scaler.fit_transform(x_train_drop[num_col])
x_test_drop[num_col]  = scaler.transform(x_test_drop[num_col])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies  = pd.get_dummies(x_test_drop)
x_test_dummies  = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

x_train_dummies = x_train_dummies.fillna(x_train_dummies.mean())
x_test_dummies  = x_test_dummies.fillna(x_train_dummies.mean())

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
models = {
    "RandomForest" : RandomForestClassifier(random_state = 23, class_weight = "balanced"),
    "GradientBoosting" : GradientBoostingClassifier(random_state = 23),
    "lightGBM" : LGBMClassifier(random_state = 23, class_weight = "balanced")
}

from sklearn.metrics import accuracy_score, f1_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    accuracy = accuracy_score(Y_valid, pred)
    f1 = f1_score(Y_valid, pred)
    print(f"{name} | Accuracy : {accuracy:.4f}, F1-score : {f1:.4f}")

RandomForest | Accuracy : 0.7784, F1-score : 0.6306
GradientBoosting | Accuracy : 0.7946, F1-score : 0.6667
[LightGBM] [Info] Number of positive: 159, number of negative: 270
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000713 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 531
[LightGBM] [Info] Number of data points in the train set: 429, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

In [16]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [17]:
best_estimator = GradientBoostingClassifier(random_state = 23)
scoring = "accuracy"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.7, 0.8, 1.0],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator,
    param_dist,
    scoring = scoring,
    cv = 3,
    n_iter = 5,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("best param :", random_search.best_params_)

best param : {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.05}


In [18]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

accuracy_train = accuracy_score(Y_train, pred_train)
accuracy_valid = accuracy_score(Y_valid, pred_valid)

f1_train = f1_score(Y_train, pred_train)
f1_valid = f1_score(Y_valid, pred_valid)

print("accuracy score(train) :", accuracy_train, "accuracy score(valid) :", accuracy_valid)
print("f1 score(train) :", f1_train, "f1 score(valid) :", f1_valid)

accuracy score(train) : 0.8904428904428905 accuracy score(valid) : 0.8054054054054054
f1 score(train) : 0.8438538205980066 f1 score(valid) : 0.6666666666666666


In [19]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

,pred
0,0
1,0
2,0
3,1
4,0
...,...
149,1
150,0
151,0
152,1


# 회귀

## 학생성적 예측 데이터
데이터 설명 : 학생성적 예측 (종속변수 :G3)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/X_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/y_test.csv  
데이터 출처 :https://www.kaggle.com/datasets/ishandutta/student-performance-data-set (참고, 데이터 수정)

In [20]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/X_test.csv")

display(x_train.head())
display(y_train.head())

,StudentID,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,...,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2
0,1714,GP,F,18,U,GT3,T,4,3,other,...,no,4,3,3,1,1,3,0,14,13
1,1254,GP,F,17,U,GT3,T,4,3,health,...,yes,4,4,3,1,3,4,0,13,15
2,1639,GP,F,16,R,GT3,T,4,4,health,...,no,2,4,4,2,3,4,6,10,11
3,1118,GP,M,16,U,GT3,T,4,4,services,...,no,5,3,3,1,3,5,0,15,13
4,1499,GP,M,19,U,GT3,T,3,2,services,...,yes,4,5,4,1,1,4,0,5,0


,StudentID,G3
0,1714,14
1,1254,15
2,1639,11
3,1118,13
4,1499,0


In [24]:
drop_col = ["StudentID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["G3"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
column = x_train_drop.select_dtypes(exclude = object).columns
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [25]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
models = {
    "RandomForest" : RandomForestRegressor(random_state = 23),
    "GradientBoosting" : GradientBoostingRegressor(random_state = 23),
    "lightGBM" : LGBMRegressor(random_state = 23)
}

from sklearn.metrics import root_mean_squared_error, r2_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    rmse = root_mean_squared_error(Y_valid, pred)
    r2 = r2_score(Y_valid, pred)
    print(f"{name} | RMSE : {rmse:.4f}, R2 : {r2:.4f}")

RandomForest | RMSE : 1.5535, R2 : 0.8299
GradientBoosting | RMSE : 1.4426, R2 : 0.8533
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000654 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 211
[LightGBM] [Info] Number of data points in the train set: 474, number of used features: 58
[LightGBM] [Info] Start training from score 11.409283
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

In [28]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [27]:
best_estimator = LGBMRegressor(random_state = 23)
scoring = "neg_root_mean_squared_error"

param_dist = {
    "n_estimator" : [100, 200, 300, 500],
    "max_depth" : [3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample" : [0.8, 0.9, 1]
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator, 
    param_dist,
    scoring = scoring,
    n_iter = 5,
    cv = 3,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("Best Params : ", random_search.best_params_)

[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000860 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 202
[LightGBM] [Info] Number of data points in the train set: 316, number of used features: 55
[LightGBM] [Info] Start training from score 11.430380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

In [29]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

rmse_train = root_mean_squared_error(Y_train, pred_train)
rmse_valid = root_mean_squared_error(Y_valid, pred_valid)

r2_train = r2_score(Y_train, pred_train)
r2_valid = r2_score(Y_valid, pred_valid)

print("RMSE(train) :", rmse_train, "RMSE(valid) :", rmse_valid)
print("R2 score(train) :", r2_train, "R2 score(valid) :", r2_valid)

[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Warning] Unknown parameter: n_estimator
RMSE(train) : 1.121157541830054 RMSE(valid) : 1.363129957978571
R2 score(train) : 0.91535030571962 R2 score(valid) : 0.8690149850212064


In [30]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

[LightGBM] [Warning] Unknown parameter: n_estimator


,pred
0,16.245025
1,10.558124
2,14.425870
3,7.410659
4,10.925014
...,...
361,15.138414
362,12.553686
363,16.007130
364,1.437003


## ⚠️ 중고차 가격 예측 데이터
데이터 설명 : 중고차 가격 예측 데이터 (종속변수 :G3)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/X_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/y_test.csv  
데이터 출처 :https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes?select=vw.csv (참고, 데이터 수정)

In [31]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/X_test.csv")

display(x_train.head())
display(y_train.head())

,carID,brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize
0,13207,hyundi,Santa Fe,2019,Semi-Auto,4223,Diesel,145.0,39.8,2.2
1,17314,vauxhall,GTC,2015,Manual,47870,Diesel,125.0,60.1,2.0
2,12342,audi,RS4,2019,Automatic,5151,Petrol,145.0,29.1,2.9
3,13426,vw,Scirocco,2016,Automatic,20423,Diesel,30.0,57.6,2.0
4,16004,skoda,Scala,2020,Semi-Auto,3569,Petrol,145.0,47.1,1.0


,carID,price
0,13207,31995
1,17314,7700
2,12342,58990
3,13426,12999
4,16004,16990


In [34]:
drop_col = ["carID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["price"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
column = x_train_drop.select_dtypes(exclude = object).columns
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [35]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
models = {
    "RandomForest" : RandomForestRegressor(random_state = 23),
    "GradientBoosting" : GradientBoostingRegressor(random_state = 23),
    "lightGBM" : LGBMRegressor(random_state = 23)
}

from sklearn.metrics import root_mean_squared_error, r2_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    rmse = root_mean_squared_error(Y_valid, pred)
    r2 = r2_score(Y_valid, pred)
    print(f"{name} | RMSE : {rmse:.4f}, R2 : {r2:.4f}")

RandomForest | RMSE : 3527.2224, R2 : 0.9547
GradientBoosting | RMSE : 4464.5500, R2 : 0.9275
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000328 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 571
[LightGBM] [Info] Number of data points in the train set: 3472, number of used features: 75
[LightGBM] [Info] Start training from score 23271.952765
lightGBM | RMSE : 4040.0501, R2 : 0.9406


In [36]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [47]:
best_estimator = RandomForestRegressor(random_state = 23)
scoring = "neg_root_mean_squared_error"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [3, 4, 5],
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator, 
    param_dist,
    scoring = scoring,
    n_iter = 5,
    cv = 3,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("Best Params : ", random_search.best_params_)

Best Params :  {'n_estimators': 500, 'max_depth': 5}


In [42]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

rmse_train = root_mean_squared_error(Y_train, pred_train)
rmse_valid = root_mean_squared_error(Y_valid, pred_valid)

r2_train = r2_score(Y_train, pred_train)
r2_valid = r2_score(Y_valid, pred_valid)

print("RMSE(train) :", rmse_train, "RMSE(valid) :", rmse_valid)
print("R2 score(train) :", r2_train, "R2 score(valid) :", r2_valid)

RMSE(train) : 5675.204627634887 RMSE(valid) : 6349.351987737662
R2 score(train) : 0.8791455365624372 R2 score(valid) : 0.8533727845334007


In [43]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

,pred
0,39674.124150
1,28838.743436
2,51455.927223
3,17763.693574
4,44004.704671
...,...
2667,43252.054929
2668,21212.193439
2669,19548.464004
2670,18566.649983


## 의료 비용 예측 데이터
데이터 설명 : 의료비용 예측문제 (종속변수 :charges)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/y_test.csv  
데이터 출처 :https://www.kaggle.com/mirichoi0218/insurance/code(참고, 데이터 수정)

In [48]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,age,sex,bmi,children,smoker,region
0,2,35,female,35.860,2,no,southeast
1,3,28,female,23.845,2,no,northwest
2,4,23,female,32.780,2,yes,southeast
3,6,52,female,25.300,2,yes,southeast
4,7,63,male,39.800,3,no,southwest


,ID,charges
0,2,5836.52040
1,3,4719.73655
2,4,36021.01120
3,6,24667.41900
4,7,15170.06900


In [50]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["charges"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
column = x_train_drop.select_dtypes(exclude = object).columns
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [51]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
models = {
    "RandomForest" : RandomForestRegressor(random_state = 23),
    "GradientBoosting" : GradientBoostingRegressor(random_state = 23),
    "lightGBM" : LGBMRegressor(random_state = 23)
}

from sklearn.metrics import root_mean_squared_error, r2_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    rmse = root_mean_squared_error(Y_valid, pred)
    r2 = r2_score(Y_valid, pred)
    print(f"{name} | RMSE : {rmse:.4f}, R2 : {r2:.4f}")

RandomForest | RMSE : 5118.6511, R2 : 0.8251
GradientBoosting | RMSE : 4877.6582, R2 : 0.8412
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000298 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 316
[LightGBM] [Info] Number of data points in the train set: 749, number of used features: 11
[LightGBM] [Info] Start training from score 13272.514101
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [52]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [53]:
best_estimator = GradientBoostingRegressor(random_state = 23)
scoring = "neg_root_mean_squared_error"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample": [0.8, 0.9, 1]
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator, 
    param_dist,
    scoring = scoring,
    n_iter = 5,
    cv = 3,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("Best Params : ", random_search.best_params_)

Best Params :  {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.01}


In [54]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

rmse_train = root_mean_squared_error(Y_train, pred_train)
rmse_valid = root_mean_squared_error(Y_valid, pred_valid)

r2_train = r2_score(Y_train, pred_train)
r2_valid = r2_score(Y_valid, pred_valid)

print("RMSE(train) :", rmse_train, "RMSE(valid) :", rmse_valid)
print("R2 score(train) :", r2_train, "R2 score(valid) :", r2_valid)

RMSE(train) : 3926.582299209229 RMSE(valid) : 4804.551063129825
R2 score(train) : 0.8963806824142313 R2 score(valid) : 0.8459137196185549


In [55]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

,pred
0,6164.070177
1,3595.828909
2,8309.682766
3,4262.142451
4,12417.210530
...,...
263,12805.424859
264,12982.162119
265,3631.009850
266,34967.756151


## 킹카운티 주거지 가격예측문제 데이터
데이터 설명 : 킹카운티 주거지 가격 예측문제 (종속변수 :price)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/y_test.csv  
데이터 출처 :https://www.kaggle.com/harlfoxem/housesalesprediction (참고, 데이터 수정)

In [66]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,id,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2,8651400730,20150428T000000,3,1.00,840,5525,1.0,0,0,...,6,840,0,1969,0,98042,47.3607,-122.085,920,5330
1,3,3163600130,20150317T000000,3,1.00,1250,8000,1.0,0,0,...,7,1250,0,1956,0,98146,47.5065,-122.337,1040,6973
2,4,5045700330,20140725T000000,4,2.50,2200,6400,2.0,0,0,...,8,2200,0,2010,0,98059,47.4856,-122.156,2600,5870
3,5,1036100130,20140808T000000,3,2.50,1980,39932,2.0,0,0,...,8,1980,0,1994,0,98011,47.7433,-122.196,2610,12769
4,6,7696630080,20140506T000000,3,1.75,1690,7735,1.0,0,0,...,7,1060,630,1976,0,98001,47.3324,-122.280,1580,7503


,ID,price
0,2,191000.0
1,3,234900.0
2,4,460000.0
3,5,442000.0
4,6,197000.0


In [67]:
x_train['date'] = pd.to_datetime(x_train['date'])
x_test['date'] = pd.to_datetime(x_test['date'])

x_train["year"] = x_train["date"].dt.year
x_test["year"] = x_test["date"].dt.year

x_train["month"] = x_train["date"].dt.month
x_test["month"] = x_test["date"].dt.month

x_train["day"] = x_train["date"].dt.day
x_test["day"] = x_test["date"].dt.day

In [70]:
drop_col = ["ID", "id", "date"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["price"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
column = x_train_drop.select_dtypes(exclude = object).columns
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [71]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
models = {
    "RandomForest" : RandomForestRegressor(random_state = 23),
    "GradientBoosting" : GradientBoostingRegressor(random_state = 23),
    "lightGBM" : LGBMRegressor(random_state = 23)
}

from sklearn.metrics import root_mean_squared_error, r2_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    rmse = root_mean_squared_error(Y_valid, pred)
    r2 = r2_score(Y_valid, pred)
    print(f"{name} | RMSE : {rmse:.4f}, R2 : {r2:.4f}")

RandomForest | RMSE : 141123.5946, R2 : 0.8693
GradientBoosting | RMSE : 144212.2104, R2 : 0.8635
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000509 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2340
[LightGBM] [Info] Number of data points in the train set: 12103, number of used features: 21
[LightGBM] [Info] Start training from score 539184.736842
lightGBM | RMSE : 135606.5874, R2 : 0.8793


In [74]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [75]:
best_estimator = LGBMRegressor(random_state = 23)
scoring = "neg_root_mean_squared_error"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample": [0.8, 0.9, 1]
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator, 
    param_dist,
    scoring = scoring,
    n_iter = 5,
    cv = 3,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("Best Params : ", random_search.best_params_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000382 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2318
[LightGBM] [Info] Number of data points in the train set: 8068, number of used features: 21
[LightGBM] [Info] Start training from score 538427.375186
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

In [76]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

rmse_train = root_mean_squared_error(Y_train, pred_train)
rmse_valid = root_mean_squared_error(Y_valid, pred_valid)

r2_train = r2_score(Y_train, pred_train)
r2_valid = r2_score(Y_valid, pred_valid)

print("RMSE(train) :", rmse_train, "RMSE(valid) :", rmse_valid)
print("R2 score(train) :", r2_train, "R2 score(valid) :", r2_valid)

RMSE(train) : 113661.29828321112 RMSE(valid) : 139390.27520448636
R2 score(train) : 0.9040067623373662 R2 score(valid) : 0.8725043798848002


In [77]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

,pred
0,4.406951e+05
1,5.371098e+05
2,3.801608e+05
3,5.500293e+05
4,5.655078e+05
...,...
4318,1.851066e+06
4319,4.762016e+05
4320,2.865791e+05
4321,4.816381e+05


## 대학원 입학가능성 데이터
데이터 설명 : 대학원 입학 가능성 예측 (종속변수 :Chance of Admit)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/y_test.csv  
데이터 출처 :https://www.kaggle.com/mohansacharya/graduate-admissions(참고, 데이터 수정)

In [78]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research
0,0,67,327,114,3,3.0,3.0,9.02,0
1,1,112,321,109,4,4.0,4.0,8.68,1
2,2,495,301,99,3,2.5,2.0,8.45,1
3,3,356,317,106,2,2.0,3.5,8.12,0
4,4,250,321,111,3,3.5,4.0,8.83,1


,ID,Chance of Admit
0,0,0.61
1,1,0.69
2,2,0.68
3,3,0.73
4,4,0.77


In [80]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["Chance of Admit"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
column = x_train_drop.select_dtypes(exclude = object).columns
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [81]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
models = {
    "RandomForest" : RandomForestRegressor(random_state = 23),
    "GradientBoosting" : GradientBoostingRegressor(random_state = 23),
    "lightGBM" : LGBMRegressor(random_state = 23)
}

from sklearn.metrics import root_mean_squared_error, r2_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    rmse = root_mean_squared_error(Y_valid, pred)
    r2 = r2_score(Y_valid, pred)
    print(f"{name} | RMSE : {rmse:.4f}, R2 : {r2:.4f}")

RandomForest | RMSE : 0.0599, R2 : 0.8014
GradientBoosting | RMSE : 0.0593, R2 : 0.8052
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000036 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 267
[LightGBM] [Info] Number of data points in the train set: 280, number of used features: 8
[LightGBM] [Info] Start training from score 0.708321
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

In [82]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [83]:
best_estimator = LGBMRegressor(random_state = 23)
scoring = "neg_root_mean_squared_error"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample": [0.8, 0.9, 1]
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator, 
    param_dist,
    scoring = scoring,
    n_iter = 5,
    cv = 3,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("Best Params : ", random_search.best_params_)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000048 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 206
[LightGBM] [Info] Number of data points in the train set: 186, number of used features: 8
[LightGBM] [Info] Start training from score 0.708656
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [84]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

rmse_train = root_mean_squared_error(Y_train, pred_train)
rmse_valid = root_mean_squared_error(Y_valid, pred_valid)

r2_train = r2_score(Y_train, pred_train)
r2_valid = r2_score(Y_valid, pred_valid)

print("RMSE(train) :", rmse_train, "RMSE(valid) :", rmse_valid)
print("R2 score(train) :", r2_train, "R2 score(valid) :", r2_valid)

RMSE(train) : 0.03804519489967003 RMSE(valid) : 0.05384581439876603
R2 score(train) : 0.9300430592470935 R2 score(valid) : 0.8395911728094395


In [85]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

,pred
0,0.674890
1,0.730736
2,0.843956
3,0.551906
4,0.652848
...,...
95,0.585545
96,0.919026
97,0.640500
98,0.889488


## 레드 와인 퀄리티 예측 데이터
데이터 설명 : 레드 와인 퀄리티 예측문제 (종속변수 :quality)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/y_test.csv  
데이터 출처 :https://www.kaggle.com/uciml/red-wine-quality-cortez-et-al-2009(참고, 데이터 수정)

In [87]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,1,10.6,0.44,0.68,4.1,0.114,6.0,24.0,0.99700,3.06,0.66,13.4
1,2,7.0,0.60,0.30,4.5,0.068,20.0,110.0,0.99914,3.30,1.17,10.2
2,3,8.0,0.43,0.36,2.3,0.075,10.0,48.0,0.99760,3.34,0.46,9.4
3,4,7.9,0.53,0.24,2.0,0.072,15.0,105.0,0.99600,3.27,0.54,9.4
4,5,8.0,0.45,0.23,2.2,0.094,16.0,29.0,0.99620,3.21,0.49,10.2


,ID,quality
0,1,6
1,2,5
2,3,5
3,4,6
4,5,6


In [89]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["quality"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
column = x_train_drop.select_dtypes(exclude = object).columns
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [90]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
models = {
    "RandomForest" : RandomForestRegressor(random_state = 23),
    "GradientBoosting" : GradientBoostingRegressor(random_state = 23),
    "lightGBM" : LGBMRegressor(random_state = 23)
}

from sklearn.metrics import root_mean_squared_error, r2_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    rmse = root_mean_squared_error(Y_valid, pred)
    r2 = r2_score(Y_valid, pred)
    print(f"{name} | RMSE : {rmse:.4f}, R2 : {r2:.4f}")

RandomForest | RMSE : 0.6435, R2 : 0.3952
GradientBoosting | RMSE : 0.6438, R2 : 0.3946
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000329 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 887
[LightGBM] [Info] Number of data points in the train set: 895, number of used features: 11
[LightGBM] [Info] Start training from score 5.610056
lightGBM | RMSE : 0.6675, R2 : 0.3493


In [91]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [92]:
best_estimator = RandomForestRegressor(random_state = 23)
scoring = "neg_root_mean_squared_error"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [3, 4, 5]
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator, 
    param_dist,
    scoring = scoring,
    n_iter = 5,
    cv = 3,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("Best Params : ", random_search.best_params_)

Best Params :  {'n_estimators': 500, 'max_depth': 5}


In [93]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

rmse_train = root_mean_squared_error(Y_train, pred_train)
rmse_valid = root_mean_squared_error(Y_valid, pred_valid)

r2_train = r2_score(Y_train, pred_train)
r2_valid = r2_score(Y_valid, pred_valid)

print("RMSE(train) :", rmse_train, "RMSE(valid) :", rmse_valid)
print("R2 score(train) :", r2_train, "R2 score(valid) :", r2_valid)

RMSE(train) : 0.5029512140549625 RMSE(valid) : 0.6488465141861862
R2 score(train) : 0.6034412935210683 R2 score(valid) : 0.38508616358462044


In [94]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

,pred
0,5.553002
1,6.480147
2,5.106601
3,4.102680
4,5.117119
...,...
315,5.211503
316,5.384113
317,5.077897
318,5.167710


## 현대 차량 가격 분류문제 데이터
데이터 설명 : 현대 차량가격 분류문제 (종속변수 :price)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/y_test.csv  
데이터 출처 :https://www.kaggle.com/mysarahmadbhat/hyundai-used-car-listing(참고, 데이터 수정)

In [95]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,model,year,transmission,mileage,fuelType,tax(£),mpg,engineSize
0,0,I30,2019,Manual,21,Petrol,150,34.0,2.0
1,1,Santa Fe,2018,Semi-Auto,10500,Diesel,145,39.8,2.2
2,2,Tucson,2017,Manual,29968,Diesel,30,61.7,1.7
3,3,Kona,2018,Manual,27317,Petrol,145,52.3,1.0
4,4,Tucson,2018,Semi-Auto,31459,Diesel,145,57.7,1.7


,ID,price
0,0,23995
1,1,28490
2,2,13251
3,3,14990
4,4,17591


In [96]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["price"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
column = x_train_drop.select_dtypes(exclude = object).columns
x_train_drop[column] = scaler.fit_transform(x_train_drop[column])
x_test_drop[column] = scaler.transform(x_test_drop[column])

x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [97]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
models = {
    "RandomForest" : RandomForestRegressor(random_state = 23),
    "GradientBoosting" : GradientBoostingRegressor(random_state = 23),
    "lightGBM" : LGBMRegressor(random_state = 23)
}

from sklearn.metrics import root_mean_squared_error, r2_score
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_valid)
    rmse = root_mean_squared_error(Y_valid, pred)
    r2 = r2_score(Y_valid, pred)
    print(f"{name} | RMSE : {rmse:.4f}, R2 : {r2:.4f}")

RandomForest | RMSE : 1243.4967, R2 : 0.9581
GradientBoosting | RMSE : 1289.4968, R2 : 0.9549
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000265 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 391
[LightGBM] [Info] Number of data points in the train set: 2721, number of used features: 22
[LightGBM] [Info] Start training from score 12689.927233
lightGBM | RMSE : 1176.2216, R2 : 0.9625


In [98]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

In [99]:
best_estimator = LGBMRegressor(random_state = 23)
scoring = "neg_root_mean_squared_error"

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [3, 4, 5],
    "learning_rate" : [0.01, 0.05],
    "subsample": [0.8, 0.9, 1]
}

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    best_estimator, 
    param_dist,
    scoring = scoring,
    n_iter = 5,
    cv = 3,
    random_state = 23
)

random_search.fit(X_train, Y_train)
print("Best Params : ", random_search.best_params_)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000061 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 387
[LightGBM] [Info] Number of data points in the train set: 1814, number of used features: 22
[LightGBM] [Info] Start training from score 12769.558434
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [100]:
best_model = random_search.best_estimator_

pred_train = best_model.predict(X_train)
pred_valid = best_model.predict(X_valid)

rmse_train = root_mean_squared_error(Y_train, pred_train)
rmse_valid = root_mean_squared_error(Y_valid, pred_valid)

r2_train = r2_score(Y_train, pred_train)
r2_valid = r2_score(Y_valid, pred_valid)

print("RMSE(train) :", rmse_train, "RMSE(valid) :", rmse_valid)
print("R2 score(train) :", r2_train, "R2 score(valid) :", r2_valid)

RMSE(train) : 1135.8398915668188 RMSE(valid) : 1234.3289793312363
R2 score(train) : 0.9615280717055613 R2 score(valid) : 0.9586790725805007


In [101]:
pred = best_model.predict(x_test_dummies)
df = pd.DataFrame({
    "pred":pred
})

df

,pred
0,8348.296623
1,6650.805505
2,34799.563066
3,13156.287155
4,10425.124681
...,...
967,2294.727083
968,20219.789104
969,3656.994889
970,5983.243489
